In [ ]:
from functools import cache

import numpy as np
from llvmlite.ir import *
from numba import *
from numba import core, extending, types


@type.__call__
class dbg:
    f = lambda s, x: x

    def __call__(s, f):
        s.f = f
        return s

    def __matmul__(s, x):
        x = s.f(x)
        s.f = lambda x: x
        print(type(x), x)
        return x


def size(a):
    return getattr(a, "size", 0) or getattr(a, "count", 0) or size(a.pointee)


def load(build, a, B):
    if not a.type.is_pointer:
        print(f"[wrn] in got {a.type} instead of ptr")
        t = build.alloca(a.type)
        build.store(a, t)
        a = t
    return build.load(build.bitcast(a, B.as_pointer()))


def store(build, a, b):
    assert b.type.is_pointer, f"out got {b.type} instead of ptr"
    if a.type.is_pointer:
        a = build.load(a)
    build.store(a, build.bitcast(b, a.type.as_pointer()))


@cache
def hpo(F):
    @extending.intrinsic
    def f(ctx, ta, tb, to):
        def gen(ctx, build: IRBuilder, sig, a):
            store(build, F(build, *(build.zext(load(build, a, IntType(size(a.type) * 8)), IntType(size(to) * 8)) for a in a[:2])), a[2])

        return void(ta, tb, to), gen

    @vectorize(nopython=True)
    def g(a, b, o):
        f(a, b, o)
        return np.bool_(0)

    return g


@cache
def hp(F):
    @extending.intrinsic
    def f(ctx, ta, tb):
        def gen(ctx, build: IRBuilder, sig, a):
            return load(build, F(build, *(build.zext(load(build, a, IntType(size(a.type) * 8)), IntType(size(sig.return_type) * 8)) for a in a)), ctx.get_value_type(sig.return_type))

        return types.CharSeq((int.__add__ if F == IRBuilder.mul else max)(size(ta), size(tb)))(ta, tb), gen

    @vectorize(nopython=True)
    def g(a, b):
        return f(a, b)

    return g

In [ ]:
a = np.array([2**16 - 1] * 2, "L")
b = np.array([2**16 - 1] * 2, "L")
o = np.zeros(1, "4L")
hpo(IRBuilder.mul)(a.view("S0,S16"), b.view("S0,S16"), o.view("S0,S32"))
o

In [ ]:
(a.astype("O") @ [1, 1 << 64]) * (b.astype("O") @ [1, 1 << 64])

In [ ]:
o.astype("O") @ [1 << i * 64 for i in range(4)]

In [ ]:
o

In [ ]:
a = np.frombuffer(np.random.bytes(32), "4L")
b = np.frombuffer(np.random.bytes(32), "4L")
o = np.empty(len(a), "8L")

In [ ]:
hpo(IRBuilder.mul)(a.view("S0,S32"), b.view("S0,S32"), o.view("S0,S64"));

In [ ]:
(a.astype("O") @ [1 << i * 64 for i in range(4)[::-1]]) * (b.astype("O") @ [1 << i * 64 for i in range(4)[::-1]]) == o.astype("O") @ [1 << i * 64 for i in range(8)[::-1]]

In [ ]:
(a.astype("O") @ [1 << i * 64 for i in range(4)[::-1]]) * (b.astype("O") @ [1 << i * 64 for i in range(4)[::-1]])

In [ ]:
o.astype("O") @ [1 << i * 64 for i in range(8)[::-1]]

In [ ]:
a: 5
b: 11
c: 13

In [ ]:
a: [5 x i8]* -> i40*

In [ ]:
a

In [ ]:
a = a[0]

In [ ]:
a.type.pointee.count

In [ ]:
x = np.frombuffer(np.random.bytes(8), "S0,S8")
y = np.frombuffer(np.random.bytes(8), "S0,S8")

In [ ]:
a[0].type.pointee.count

In [ ]:
mul(x, y)

In [ ]:
ta

In [ ]:
types.UniTuple(ta, 2).dtype.size

In [ ]:
1024 / 8

In [ ]:
def cast(build, a, t):
    t = build.alloca(t)
    build.store(t.type.pointee(0), t)
    build.store(a, build.bitcast(t, a.type.as_pointer()))
    return build.load(t)


def pcast(build, a, t):
    return build.bitcast(a, t.type.as_pointer())


# @vectorize(nopython=True)
@njit
def mul(a, b):
    return mul_(a, b)


x = np.random.randint(0, 1 << 64, (1, 4), "L").view("i8")[0]
y = np.random.randint(0, 1 << 64, (1, 4), "L").view("i8")[0]
# o = np.zeros((100, 8), "L").view("S64")
mul(x, y)

In [ ]:
build.load_reg(IntType(64))

In [ ]:
import numpy as np
from llvmlite.ir import *
from numba import *
from numba import core, extending, types


@extending.intrinsic
def mul(ctx, a, b, typ=IntType(1024)):
    """
    this is a function that multiplys two big decimal "u8[:]" and returns a big decimal "UniTuple(u8,16)"

    now this thing won't work because a_len and b_len are dynamic. Fix this by replacing the for loop with actual llvm ir loop instead
    """

    def restore(a: PointerType(IntType(64)), len: IntType(64), builder: IRBuilder):
        l = build.extract_value(a, [5, 1])
        a = build.extract_value(a, 4)
        o = typ(0, )

        i = IntType(64)(0)
        with build.if_then(build.icmp_unsigned("<", i, l)):
            build.add(o, build.shl(build.zext(build.load(build.gep(a, [i])), typ), build.zext(build.mul(i, 64)))

    def gen(ctx, build, sig, arg):
        a, b = arg
        a_ptr = build.extract_value(a, 4)
        a_len = build.extract_value(a, [5, 1])
        b_ptr = build.extract_value(b, 4)
        b_len = build.extract_value(b, [5, 1])

        u4 = IntType(32)
        u128 = IntType(1024)
        a_val = functools.reduce(
            build.add,
            [build.shl(build.zext(build.load(build.gep(a_ptr, [u4(i)])), u128), u128(i * 64)) for i in range(a_len)],
        )
        b_val = functools.reduce(
            build.add,
            [build.shl(build.zext(build.load(build.gep(b_ptr, [u4(i)])), u128), u128(i * 64)) for i in range(a_len)],
        )
        ab_val = build.mul(a_val, b_val)

        out = build.alloca(IntType(512))
        build.store(ab_val, out)
        return build.load(build.bitcast(out, ctx.get_value_type(sig.return_type).as_pointer()))

    return types.UniTuple(u8, 8)(a, b), gen


f(np.random.rand(4, 4))

In [ ]:
build.branch

In [ ]:
a[0].type

In [ ]:
mul(np.random.randint(0, 100, [8, 1], "L"), np.random.randint(0, 100, [8, 1], "L"))

In [ ]:
@extending.intrinsic
def tup_(ctx, a):
    print(a)

    def gen(ctx, build, sig, a):
        return build.load(build.bitcast(build.extract_value(a[0], 4), ctx.get_value_type(sig.return_type).as_pointer()))

    return types.UniTuple(a.dtype, 16)(a), gen


@njit
def tup(a):
    return tup_(a)
    # return tup_(a.tolist())


tup(np.random.rand(10))

In [ ]:
np.random.rand(4).tolist()

In [ ]:
(72057594037927936 << 64) + (144115188075855872 << 128) + (72057594037927936 << 192)

In [ ]:
((1 << 60 << 64) + (1 << 60)) ** 2

In [ ]:
@extending.intrinsic
def f_(ctx):
    def gen(ctx, build, sig, a):
        t = build.alloca(IntType(128))
        build.store(IntType(128)(41273491274982317490127409217971), t)
        return build.load(build.bitcast(t, ArrayType(IntType(64), 2).as_pointer()))

    return types.UniTuple(u8, 2)(), gen


@njit
def f():
    return f_()

In [ ]:
f()

In [ ]:
t.type.pointee

In [ ]:
5890234580342850928028490238402

In [ ]:
@extending.intrinsic
def mul_(ctx, a, b):
    def gen(ctx: core.cpu.CPUContext, build: IRBuilder, sig: core.typing.Signature, a):
        globals().update(locals())
        u8 = IntType(64)
        u16 = IntType(128)
        a = build.mul(build.zext(a[0], u16), build.zext(a[1], u16))
        o = build.alloca(ArrayType(u8, 2))
        build.store(build.trunc(a, u8), build.gep(o, [u8(0), u8(0)]))
        build.store(build.trunc(build.lshr(a, u16(64)), u8), build.gep(o, [u8(0), u8(1)]))
        return o

    return types.Tuple((types.u8, types.u8))(types.u8, types.u8), gen


@njit
def mul(a, b):
    return mul_(a, b)


x = 2**60 + 1
mul(x, x)
print(mul.inspect_llvm(mul.signatures[0]))

In [ ]:
core.typing.Signature

In [ ]:
__import__("importlib").reload(__import__("nn"))
from nn import *
from ray import train, tune
from ray.tune.search.hyperopt import HyperOptSearch


def train_rets(O, dtype=torch.bfloat16, compile=lambda m, x: torch.compile(torch.jit.trace(m, x)), report=train.report):
    a = Dat("py.txt", 32, 512)
    X = a().to("cuda", torch.long)
    m = bisect(lambda x: Rets(**mul(O, x)), size, 16, O.get("sz", 100000))[1]
    m = compile(m.to("cuda", dtype), X)
    o = torch.optim.Adam(m.parameters(), **O.get("o", {}))
    O = torch.optim.lr_scheduler.OneCycleLR(o, **dict(max_lr=0.01, total_steps=10000, pct_start=0.01, div_factor=3, final_div_factor=9) | O.get("O", {}))
    L = A = 0
    for _ in range(1, 1 + O.total_steps):
        x, X = X, a().to("cuda", torch.long)
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y.mT, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.data.log()
        A += (y.argmax(2) != x).sum().log()
        if not _ % 100:
            report({"l": float((L / 100).exp()), "a": 1 - float((A / 100).exp()) / x.numel()})
            L = A = 0

In [ ]:
Mixs([{"i": 4}, {"i": 68}])

In [ ]:
ApLeft(5, 6, 7, 1)

In [ ]:
AvgPool1d(10, padding=9)(torch.rand(2, 3, 4))

In [ ]:
ConvLeft
Pool:
    AvgPool
    MaxPool
SE

In [ ]:
import matplotlib.pyplot as plt


# @torch.compile(dynamic=True)
def asort(x, y):
    for i in range(len(y) - 1):
        for j in range(len(y) - 1 - i):
            if x[y[j]] > x[y[j + 1]]:
                y[j], y[j + 1] = y[j + 1], y[j]


# @torch.compile(dynamic=True)
def qsort(x, y):
    if len(y) <= 256:
        asort(x, y)
        return
    p = x[y[len(y) // 2]]
    i = 0
    j = len(y) - 1
    while i <= j:
        while x[y[i]] < p:
            i += 1
        while x[y[j]] > p:
            j -= 1
        if i < j:
            y[i], y[j] = y[j], y[i]
        i += 1
        j -= 1
    qsort(x, y[:i])
    qsort(x, y[j + 1 :])


x = torch.rand(1024)
y = torch.arange(1024)
qsort(x, y)
plt.plot(x[y])

In [ ]:
plt.plot(torch.rand(1024))

In [ ]:
comp = torch.jit.trace(a, x)

In [ ]:
comp(torch.rand(8, 16, dtype=torch.bfloat16, device="cuda"))